In [1]:
import sys
sys.path.append("..")  # Adds the parent directory (src) to the Python path

In [2]:
from backend.backend.utils import pod_parser

In [4]:
import requests
#SERVER_URL = "http://127.0.0.1:8008"
SERVER_URL = "http://192.168.2.239"
# gather the podcast slug, episode guid, and audio file link for transcription
podcasts = requests.get(f"{SERVER_URL}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']} ({p['slug']})")

0: Verdict with Ted Cruz (verdict-with-ted-cruz)
1: Leger om livet (leger-om-livet)
2: Norsken, svensken og dansken (norsken-svensken-og-dansken)
3: Huberman Lab (huberman-lab)
4: The Ben Shapiro Show (the-ben-shapiro-show)
5: The Megyn Kelly Show (the-megyn-kelly-show)
6: The Daily (the-daily)
7: Checks and Balance from The Economist (checks-and-balance-from-the-economist)
8: Pod Save America (pod-save-america)
9: Logbuch:Netzpolitik (logbuchnetzpolitik)
10: Apokalypse & Filterkaffee (apokalypse-filterkaffee)
11: Lage der Nation - der Politik-Podcast aus Berlin (lage-der-nation-der-politik-podcast-aus-berlin)
12: Inside Europe | Deutsche Welle (inside-europe-deutsche-welle)
13: Forklart (forklart)
14: Oppdatert (oppdatert)
15: USApodden (usapodden)
16: Det Store Bildet (det-store-bildet)
17: Best of the Left - Progressive Politics and Culture, Curated by Humans, Not Algorithms (best-of-the-left-progressive-politics-and-culture)
18: Gaslit Nation with Andrea Chalupa and Sarah Kendzior 

In [ ]:
def get_new_episodes(podcast):
    """Get the new episodes of a podcast from the RSS feed.

    Parameters
    ----------
    podcast : dict
        The podcast dictionary.

    Returns
    -------
    list
        A list of new episodes.
    """
    episodes_all = pod_parser.parse_channel(podcast["rss"])["audioitem_set"]
    episodes_db = [entry["guid"] for entry in podcast["audioitem_set"]]
    new_eps = []
    for episode in episodes_all:
        if episode["guid"] not in episodes_db:
            new_eps.append(episode)
        else:
            break
    return new_eps



In [ ]:
def get_episode_by_guid(podcast_slug, podcasts, episode_guid):
    """return an episode of the podcast with the given guid by querying the server and reading the RSS feed
    
    Parameters
    ----------
    podcast_slug : str
        The slug of the podcast.
    podcasts : list
        List of podcast objects.
    episode_guid : str
        The guid of the episode to add.

    Returns
    -------
    dict
        The episode dictionary.
    """
    podcast = [pod for pod in podcasts if pod["slug"] == podcast_slug][0]
    episodes_all = pod_parser.parse_channel(podcast["rss"])
    for episode in episodes_all["audioitem_set"]:
        if episode["guid"] == episode_guid:
            return episode
    return None


In [ ]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy':
                    segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)

In [ ]:
def get_episodes_wo_transcription(podcasts):
    """Get the episodes already in the database without transcription for each podcast.

    Parameters
    ----------
    podcasts : list
        A list of podcasts.

    Returns
    -------
    list
        A list of episodes without transcription for each podcast.
    """
    episodes_wo_transcription = []
    for podcast in podcasts:
        for audioitem in podcast['audioitem_set']:
            # check if audioitem['transcription_set'] is empty
            if len(audioitem['transcription_set']) == 0:
                episodes_wo_transcription.append((audioitem, podcast))
    return episodes_wo_transcription


In [ ]:
import os

def download_audio_file(new_ep, podcast_slug):
    current_dir = os.path.dirname(os.getcwd()) # get parent of current directory
    media_dir = current_dir + "/media"

    link = new_ep.get("audio_link")
    guid = new_ep.get("guid")
    fileroot = f"{media_dir}/{podcast_slug}_{guid}"
    # check for wav file
    if not os.path.exists(f"{fileroot}.wav"):
        # check for mp3 file
        if not os.path.exists(f"{fileroot}.mp3"):
            # download mp3 file
            print(f"Downloading {fileroot}")
            !curl -o '{fileroot + ".mp3"}' -L -J '{link}'
        # convert mp3 to wav
        !ffmpeg -i '{fileroot + ".mp3"}' -vn -acodec pcm_s16le -ar 16000 -ac 1 '{fileroot + ".wav"}'
        print(f"Downloaded {fileroot}")
    else:
        print(f"File {fileroot} already exists")


    filepath = fileroot + ".wav"
    return filepath

In [ ]:
def add_visibility_to_utterances(utterances, visibility):
    """Add the visibility field to each utterance.

    Parameters
    ----------
    utterances : list
        A list of utterances.
    visibility : JSON, 1 for all, or a list of the visible qualifiers
        The visibility of the utterances.

    Returns
    -------
    list
        A list of utterances with the visibility field.
    """
    for utterance in utterances:
        utterance["visibility"] = visibility
    return utterances

In [ ]:
for podcast in podcasts:
    new_eps = get_new_episodes(podcast)
    if len(new_eps) > 0:
        print(f"Found {len(new_eps)} new episodes for {podcast['title']}")

### Fetch new episodes for existing podcasts

In [ ]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy

# update podcasts that are already in the database with new episodes
for podcast in podcasts:
    slug = podcast["slug"]
    print(slug)
    
    new_eps = get_new_episodes(podcast)
    files = []
    for ep in new_eps:
        print(ep.get("title"))
        # get file
        file = download_audio_file(ep, slug)
        # run transcription
        lang = podcast["language"][0:2]
        script = get_transcription(file, language=lang if lang != "nb" else "no", model_size="large")

        # post episode to api
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/episodes/", json=ep)
        print(res.status_code)

        # post transcription to api
        transcription_dict = script["transcription"]
        guid = "_".join(file.split("/")[-1].split("_")[1:]).split(".")[0]
        transcription_dict["guid"] = guid
        res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
        print(res.status_code)

        # post whisper default segmentation
        segmentation_dict = script["segmentation"]
        segmentation_dict["utterance_set"] = add_visibility_to_utterances(segmentation_dict["utterance_set"], 1)
        trans_uuid = res.json().get("uuid")
        segmentation_dict["uuid"] = trans_uuid
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
        print(res.status_code)

        # use spacy to split the text into sentences
        try :
            utterances = sentence_splitter.sentence_splitter(transcription_dict, "en_core_web_lg" if lang == "en" else "nb_core_news_lg")

            segmentation_dict_spacy = {
                "uuid": trans_uuid,
                "name": "spaCy",
                "segmentor": {"name": "spaCy", "version": spacy.__version__},
                "utterance_set": add_visibility_to_utterances([utterance for utterance in utterances if utterance.get("text") != ""], 1)
            }

            res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict_spacy)
            print("spaCy", res.status_code)
        except:
            print("spaCy failed")



### Add NRK episodes from feed on GitHub

In [ ]:
# get the latest NRK episodes from this hacked feed on github
pk_eps = pod_parser.parse_channel("https://sindrel.github.io/nrk-pod-feeds/rss/debatten.xml")["audioitem_set"]

In [ ]:
podcast = None
for pod in podcasts:
    if pod["slug"] == "debatten":
        podcast = pod
        break
podcast

In [ ]:
podcast_slug = podcast["slug"]

for ep in pk_eps:
    # post episode to api
    res = requests.post(f"{SERVER_URL}/api/podcasts/{podcast_slug}/episodes/", json=ep)
    print(res.status_code)

### Add specific episodes to existing podcast by their GUID always found in their RSS (transcription runs separately)

In [ ]:
podcast = None
for pod in podcasts:
    if pod["slug"] == "lrntech":
        podcast = pod
        break
podcast

In [ ]:
episode_guids = [
    "tagsoundcloud2010tracks1432082845",
]

In [ ]:
episode_guids = [
"644c2eef74aba2001128a6c0",
"644c2e350ace130011ac671b",
"644c2d370126970011d8b435",
"644c2bccdb63c900112e843e",
"642c6c59dcec3a00115d0a6e",
"642c6babc6ef3c00110bd5f7",
"642c6aa3be840800113180a0",
"642c68bda5f38c0011fba399",
"641f8790039ec50011b15e4e",
"641f873dd0199a001159ae79",
"641f86b0c3bc870011adc0ef",
"641f8603c3bc870011ada5f9",
"63f9fefdb926660011151401",
"63fa0086534f640011ef65f5",
"63fa00000a0d230011e4e5b3",
"63f9fe069cef060011647ceb",
]

In [ ]:
episode_guids = [
    "642c6659dcec3a00115bdbae",
"642c63efac6baa00113e55c7",
"642c60e5c6ef3c001109c750",
"642c5e7b65d917001168ae05",
]

In [ ]:
podcast_slug = podcast["slug"]

for guid in episode_guids:
    ep = get_episode_by_guid(podcast_slug, podcasts, guid)
    # post episode to api
    res = requests.post(f"{SERVER_URL}/api/podcasts/{podcast_slug}/episodes/", json=ep)
    print(res.status_code)

## Add a new podcast, and 5 latest episodes

In [ ]:
new_eps = ["https://podkast.nrk.no/program/debatten.rss"]

In [ ]:
new_eps = ["https://feeds.acast.com/public/shows/5b8387cc6f8260f2513de48e"]

In [ ]:
import requests
# Add each podcast in RSS to database via API
# the API will automatically download some number of recent episodes

for podcast in new_eps:
  print(podcast)
  res = requests.post(f"{SERVER_URL}/api/podcasts/", data={
    "rss": podcast
  })
  print(res.status_code)

In [5]:
podcast = podcasts[36]
eps = podcast["audioitem_set"]
len(eps)

58

In [8]:
eps[0]

{'title': '#C1387_230526_Miloš Novović og Frode Skaarnes: Risikoreduserende tiltak',
 'subtitle': '#C1387_230526_Miloš Novović og Frode Skaarnes: Risikoreduserende tiltak',
 'author': None,
 'link': 'https://shows.acast.com/lrntech/episodes/c1387-230526-milo-novovi-og-frode-skaarnes-risikoreduserende',
 'summary': '<p><strong>Hvordan implementerer man risikoreduserende tiltak? Og hvordan tar man en helhetsvurdering av sikkerheten til en organisasjon? Frode Skaarnes og Milos Novovic gjester denne episoden og snakker om teknologiske faktorer, menneskelige faktorer og mye mer.&nbsp;</strong></p><p><br /></p><ul><li>“Sikkerhetstiltak er ikke lenger et valgfag, men hygienefaktor”</li></ul><p><br /></p><br /><hr /><p style="color: grey; font-size: 0.75em;"> Hosted on Acast. See <a href="https://acast.com/privacy" rel="noopener noreferrer" style="color: grey;" target="_blank">acast.com/privacy</a> for more information.</p>',
 'description': None,
 'image': 'https://assets.pippa.io/shows/5b838

In [13]:


eps[0]["title"].replace("_", " ") 

'#C1387 230526 Miloš Novović og Frode Skaarnes: Risikoreduserende tiltak'

Run a different size of Whisper model

In [14]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy
from json import JSONDecodeError

# update podcasts that are already in the database with new episodes

for episode in eps:
    slug = podcast.get("slug")
    print(episode.get("title"))
    audio_path = f"../media/{podcast.get('slug')}_{episode.get('guid')}.wav" 

    # run transcription
    lang = podcast["language"][0:2]
    script = get_transcription(audio_path, language=lang if lang != "nb" else "no", model_size="medium", initial_prompt=episode["title"].replace("_", " ") )

    # post transcription to api
    transcription_dict = script["transcription"]
    guid = "_".join(audio_path.split("/")[-1].split("_")[1:]).split(".")[0]
    transcription_dict["guid"] = guid
    res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
    print(res.status_code)

    # post whisper default segmentation
    segmentation_dict = script["segmentation"]
    trans_uuid = res.json().get("uuid")
    segmentation_dict["uuid"] = trans_uuid
    segmentation_dict["agentsession_set"] = []
    for utt in segmentation_dict["utterance_set"]:
        utt["visibility"] = 1

    res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
    print(res.status_code)


#C1387_230526_Miloš Novović og Frode Skaarnes: Risikoreduserende tiltak
201
201
#C1386_230516_Miloš Novović og Frode Skaarnes: Risikovurdering
201
201
#C1385_230516_Miloš Novović og Frode Skaarnes: Generelt om sikkerhet
201
201
#C1384_230516_Miloš Novović og Lin Jacobsen Hammer: ISO-standarder
201
201
#C1433_230428_Nariman Fakhraee og Evija Izaka: Part 4- Fleksibilitet i smarte bygg
201
201
#C1432_230428_Nariman Fakhraee og Evija Izaka: Part 3- Energioptimalisering i bygg
201
201
#C1431_230428_Nariman Fakhraee og Evija Izaka: Part 2- Digitalisering og drift av bygg
201
201
#C1430_230428_Nariman Fakhraee og Evija Izaka: Part 1- Fremtidens bygg: introduksjon til tema og gjester
201
201
M0069d_230425_Bente Øverli og Nadia Ullah: Åpenhetsloven, leksjon 4 verksted
201
201
M0069c_230425_Bente Øverli og Nadia Ullah: Åpenhetsloven, leksjon 3 verktøy
201
201
M0069b_230425_Bente Øverli og Nadia Ullah: Åpenhetsloven, leksjon 2 eksempler
201
201
M0069a_230425_Bente Øverli og Nadia Ullah: Åpenhetsl

## Find Episodes that don't have a transcription and try running one

In [ ]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy
from json import JSONDecodeError

eps = get_episodes_wo_transcription(podcasts)

# update podcasts that are already in the database with new episodes

for episode in eps:
    podcast = episode[1]
    ep = episode[0]
    slug = podcast.get("slug")
    print(ep.get("title"))
    # get file
    file = download_audio_file(ep, slug)
    # run transcription
    lang = podcast["language"][0:2]
    script = get_transcription(file, language=lang if lang != "nb" else "no", model_size="large")

    # post transcription to api
    transcription_dict = script["transcription"]
    guid = "_".join(file.split("/")[-1].split("_")[1:]).split(".")[0]
    transcription_dict["guid"] = guid
    res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
    print(res.status_code)

    # post whisper default segmentation
    segmentation_dict = script["segmentation"]
    trans_uuid = res.json().get("uuid")
    segmentation_dict["uuid"] = trans_uuid
    segmentation_dict["agentsession_set"] = []
    for utt in segmentation_dict["utterance_set"]:
        utt["visibility"] = 1

    res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
    print(res.status_code)

    # use spacy to split the text into sentences
    spacy_dict = {"en": "en_core_web_lg", "no": "nb_core_news_lg", "de": "de_dep_news_trf", "se": "sv_core_news_lg", "da": "da_core_news_trf"}
    try :
        utterances = sentence_splitter.sentence_splitter(transcription_dict, spacy_dict[lang])
        
        for utt in utterances:
            utt["visibility"] = 1

        segmentation_dict_spacy = {
            "uuid": trans_uuid,
            "name": "spaCy",
            "segmentor": {"name": "spaCy", "version": spacy.__version__},
            "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""],
            "agentsession_set": [],
            "visibility": 1, # SET ON UTTERANCE NOT SEGMENTATION!!!!
        }

        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict_spacy)
        print("spaCy", res.status_code)
    except:
        print("spaCy failed")

In [ ]:
for key, value in transcription_dict.items():
    print(key, value, type(value))